> The monitoring data and building-specific configuration are confidential and are not included. See `README.md` for execution requirements. Clear any generated outputs before sharing an executed notebook.


In [ ]:
# Private workspace and configuration
from copy import deepcopy as _privacy_deepcopy
from functools import lru_cache as _privacy_lru_cache
import json
import os
from pathlib import Path


def _private_config_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} before running this notebook; see README.md.")
    return Path(value).expanduser().resolve()


def private_workspace():
    """Return the configured private input/output directory."""
    path = _private_config_path("HVAC_PRIVATE_WORKSPACE")
    if not path.is_dir():
        raise FileNotFoundError("The configured private workspace does not exist.")
    return path


def _decode_private_values(value):
    if isinstance(value, list):
        return [_decode_private_values(item) for item in value]
    if isinstance(value, dict):
        if set(value) == {"__tuple_key_dict__"}:
            return {tuple(key): _decode_private_values(item)
                    for key, item in value["__tuple_key_dict__"]}
        return {key: _decode_private_values(item) for key, item in value.items()}
    return value


@_privacy_lru_cache(maxsize=1)
def _load_private_values():
    with _private_config_path("HVAC_PRIVATE_SETTINGS").open(encoding="utf-8") as handle:
        values = json.load(handle).get("private_values")
    if not isinstance(values, dict):
        raise ValueError("Private settings must contain a private_values object.")
    return _decode_private_values(values)


def private_value(key):
    """Read a private setting without embedding its value in the public source."""
    values = _load_private_values()
    if key not in values or values[key] is None:
        raise KeyError(f"Required private setting is missing: {key!r}.")
    return _privacy_deepcopy(values[key])


# Notebook 03: Remaining-duration prediction and evaluation

This notebook compares Direct ET, Temperature ET and the Shared NN on the chronological test partition from Notebook 01. Direct ET predicts remaining duration directly. The two trajectory methods reconstruct future temperatures and calculate completion time using the same evaluation procedure.

The primary comparison uses identical test prediction times at which every model returns a finite estimate. Full neural coverage is reported separately. The code calls each prediction time an **anchor**; its input-row timestamp may precede an off-grid event start. Duration calculations account for that timestamp gap.

Errors are reported in minutes and relative to observed remaining duration:

\[
\mathrm{RAE}_i=100\frac{|\widehat{\tau}_i-\tau_i|}{\max(\tau_i,5\ \mathrm{min})}.
\]

The five-minute denominator floor limits the influence of near-zero targets. Additional diagnostics group errors by room, remaining duration and trajectory regularity \(q\), and examine temperature/rate accuracy near event completion. The diagnostic score is not a model input.

Future-trajectory inference is batched over test cases. Each forecast holds the current input context fixed while querying successive horizons. This evaluates recorded operation with the offline inputs prepared in Notebook 01; it does not test a scheduling controller or measure energy and comfort outcomes.


In [ ]:

# Imports, persistent reference training run, paths

import os,re,json,math,warnings,hashlib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from scipy.stats import spearmanr

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

import torch
import torch.nn as nn

try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
except Exception: pass

def safe_torch_load(path,map_location=None):
    try:
        return torch.load(path,map_location=map_location,weights_only=False)
    except TypeError:  # compatibility with older PyTorch
        return torch.load(path,map_location=map_location)

BASE_DIR=Path(str(private_workspace()))
DATA_DIR=BASE_DIR/'full_rampup_5min_feature_dataset_UNIFIED_DERIVATIVE_DURATION_AWARE'
TRAINING_ROOT=BASE_DIR/'training_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL'
TRAINING_RUN_DIR=TRAINING_ROOT/'paper_final_q_weighted'
TRAIN_RUN=Path(TRAINING_RUN_DIR)
if not TRAIN_RUN.exists(): raise FileNotFoundError(f'Run final Notebook 02 first: {TRAIN_RUN}')
TRAIN_MODEL_DIR=TRAIN_RUN/'models'
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_ROOT=BASE_DIR/'duration_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL'/RUN_ID
PRED_DIR=OUT_ROOT/'predictions'; TABLE_DIR=OUT_ROOT/'tables'; FIG_DIR=OUT_ROOT/'figures'; MODEL_DIR=OUT_ROOT/'models'; AUDIT_DIR=OUT_ROOT/'audit'
for p in [PRED_DIR,TABLE_DIR,FIG_DIR,MODEL_DIR,AUDIT_DIR]: p.mkdir(parents=True,exist_ok=True)

PHASES=['heating','cooling']; STEP_MIN=5.0; END_BAND_C=0.4
INDIRECT_MAX_HORIZON_BY_PHASE={'heating':185.0,'cooling':65.0}
PAPER_TEMPERATURE_XMAX_BY_PHASE={'heating':180.0,'cooling':60.0}
EXTRAPOLATE_FINAL_N_POINTS=4; EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN=1e-4
EXTRAPOLATE_MAX_EXTRA_MIN_BY_PHASE={'heating':60.0,'cooling':30.0}
DIRECT_ET_PARAMS=dict(n_estimators=600,min_samples_leaf=2,max_features=0.8,random_state=42,n_jobs=-1)
MIN_DIRECT_ET_TRAIN_ANCHORS=20
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cuda': torch.set_float32_matmul_precision('high')
print('Using PAPER-FINAL training run:',TRAIN_RUN); print('DEVICE:',DEVICE); print('Output:',OUT_ROOT)


In [ ]:

# Load saved duration anchors + shared split

def norm_event_id(x):
    if pd.isna(x): raise RuntimeError('Missing event_id')
    s=str(x).strip()
    if re.fullmatch(r'[-+]?\d+\.0+',s): s=str(int(float(s)))
    return s

split=pd.read_csv(DATA_DIR/'shared_event_split_map.csv')
split['event_id']=split['event_id'].map(norm_event_id); split['room']=split['room'].astype(str).str.strip()
split['event_key']=split['phase'].astype(str)+'|'+split['room']+'|'+split['event_id']
split=split[['event_key','split','split_block']].drop_duplicates()

frames=[]
for phase in PHASES:
    p=DATA_DIR/phase/f'{phase}_remaining_duration_anchors_5min.csv'
    d=pd.read_csv(p); d['phase']=phase
    for c in ['timestamp','anchor_timestamp','feature_source_timestamp','rampup_start_time','rampup_end_time','rampup_end_grid_time']:
        if c in d.columns: d[c]=pd.to_datetime(d[c],errors='coerce')
    d['event_id']=d['event_id'].map(norm_event_id); d['room']=d['room'].astype(str).str.strip()
    d['event_key']=d['phase']+'|'+d['room']+'|'+d['event_id']
    d=d.merge(split,on='event_key',how='left',validate='many_to_one')
    if d['split'].isna().any(): raise RuntimeError(f'{phase}: unmatched duration anchors in shared split.')
    d['true_remaining_duration_min']=pd.to_numeric(d.get('true_remaining_duration_min',d.get('remaining_duration_min')),errors='coerce')
    d['remaining_trajectory_quality_score']=pd.to_numeric(d['remaining_trajectory_quality_score'],errors='coerce').clip(0,1)
    d['asof_gap_min']=(d['anchor_timestamp']-d['feature_source_timestamp']).dt.total_seconds()/60.0
    d['asof_gap_min']=d['asof_gap_min'].clip(lower=0.0)
    anchor_kind=(d['anchor_kind'] if 'anchor_kind' in d.columns else pd.Series('anchor', index=d.index)).astype(str)
    d['anchor_key']=d['event_key']+'|'+d['anchor_timestamp'].astype(str)+'|'+anchor_kind
    # Event-end threshold common form: target - sign*0.4 C.
    d['thermal_mode_sign']=pd.to_numeric(d['thermal_mode_sign'],errors='coerce')
    d['Duration_Crossing_Threshold_C']=pd.to_numeric(d['Phase_Target_T_C'],errors='coerce')-d['thermal_mode_sign']*END_BAND_C
    frames.append(d)
ANCHORS=pd.concat(frames,ignore_index=True,sort=False)

required=['Room_T','Phase_Target_T_C','thermal_mode_sign','true_remaining_duration_min','feature_source_timestamp','anchor_timestamp','remaining_trajectory_quality_score']
miss=[c for c in required if c not in ANCHORS.columns]
if miss: raise RuntimeError(f'Missing new-framework anchor columns: {miss}')
print('Anchors:',len(ANCHORS),'events:',ANCHORS['event_key'].nunique())
display(ANCHORS.groupby(['phase','split']).agg(anchors=('anchor_key','nunique'),events=('event_key','nunique'),rooms=('room','nunique')))

ANCHORS['quality_bin']=pd.cut(ANCHORS['remaining_trajectory_quality_score'],[-1e-9,.25,.5,.75,1.000001],labels=['q<=0.25','0.25<q<=0.50','0.50<q<=0.75','q>0.75'])
score_counts=ANCHORS.groupby(['phase','split','quality_bin'],observed=True).size().rename('anchors').reset_index()
score_counts.to_csv(TABLE_DIR/'duration_quality_score_bin_counts.csv',index=False)
print('\nTrajectory-regularity score-bin counts:'); display(score_counts)

# Saved split fingerprint — must match Notebook 02 exactly.
def shared_split_sha256(path):
    z=pd.read_csv(path); z['room']=z['room'].astype(str).str.strip(); z['event_id']=z['event_id'].map(norm_event_id)
    cols=['phase','room','event_id','split','split_block']; z=z[cols].drop_duplicates().sort_values(['phase','room','event_id']).reset_index(drop=True)
    return hashlib.sha256(z.to_csv(index=False,lineterminator='\n').encode('utf-8')).hexdigest()
CURRENT_SPLIT_SHA256=shared_split_sha256(DATA_DIR/'shared_event_split_map.csv')
print('Evaluation shared split SHA256:',CURRENT_SPLIT_SHA256)


In [ ]:

# Load reference preprocessor/model + assert split identity + ET artifacts

PREP=joblib.load(TRAIN_MODEL_DIR/'unified_numeric_preprocessor.joblib'); MODEL_INPUTS=PREP['numeric_cols']
manifest_path=TRAIN_RUN/'training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'
if not manifest_path.exists(): raise FileNotFoundError(manifest_path)
with open(manifest_path) as f: TRAIN_MANIFEST=json.load(f)
if TRAIN_MANIFEST.get('shared_split_sha256')!=CURRENT_SPLIT_SHA256:
    raise RuntimeError('Notebook 03 split differs from the exact split used to train Notebook 02.')
SELECTED_SCORE_MODE='score_weighted'

class UnifiedDirectedRateNN(nn.Module):
    def __init__(self,n_in,hidden=(128,64,32),dropout=0.10):
        super().__init__(); layers=[]; prev=n_in
        for h in hidden:
            layers += [nn.Linear(prev,h),nn.Tanh()]
            if dropout>0: layers.append(nn.Dropout(dropout))
            prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(-1)

ckpt=safe_torch_load(TRAIN_MODEL_DIR/'unified_directed_rate_nn.pt',map_location=DEVICE)
if ckpt.get('shared_split_sha256')!=CURRENT_SPLIT_SHA256: raise RuntimeError('Best NN checkpoint has a different shared split signature.')
NN_MODEL=UnifiedDirectedRateNN(ckpt['n_inputs'],ckpt['hidden_dims'],ckpt['dropout']).to(DEVICE); NN_MODEL.load_state_dict(ckpt['state_dict']); NN_MODEL.eval()
print('Loaded q-weighted best-validation NN from epoch:',ckpt.get('best_epoch'))

ET_ROOT=TRAIN_MODEL_DIR/'room_phase_extra_trees_temperature'; ET_ART={}
et_manifest=pd.read_csv(ET_ROOT/'model_manifest.csv')
for _,r in et_manifest.iterrows():
    p=Path(r['artifact_path']);
    if not p.exists(): p=ET_ROOT/str(r['phase'])/(str(r['room'])+'.joblib')
    ET_ART[(str(r['phase']),str(r['room']))]=joblib.load(p)

def transform_nn(df):
    x=df[MODEL_INPUTS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).copy(); phases=df['phase'].astype(str).to_numpy()
    for phase in PHASES:
        idx=np.where(phases==phase)[0]
        if len(idx)==0: continue
        med=PREP['phase_medians'][phase]
        for c in MODEL_INPUTS:
            j=x.columns.get_loc(c); x.iloc[idx,j]=x.iloc[idx,j].fillna(float(med[c])).to_numpy()
    if x.isna().any().any(): raise RuntimeError('NN transform left NaN values.')
    return PREP['scaler'].transform(x.to_numpy(float)).astype(np.float32)


In [ ]:

# Exact label-matched duration converter + source/anchor alignment

def crossed(phase,T,threshold): return float(T)>=float(threshold) if phase=='heating' else float(T)<=float(threshold)
def direction_valid(phase,dT): return np.isfinite(dT) and ((dT>0) if phase=='heating' else (dT<0))

def linear_crossing(h0,T0,h1,T1,threshold):
    if abs(T1-T0)<1e-12: return float(h1)
    a=np.clip((threshold-T0)/(T1-T0),0,1); return float(h0+a*(h1-h0))

def estimate_crossing_from_source(phase,current_T,threshold,horizons,pred_T,current_delta_prev=np.nan):
    h=np.asarray(horizons,float); y=np.asarray(pred_T,float)
    m=np.isfinite(h)&np.isfinite(y); h=h[m]; y=y[m]
    if len(h)==0: return np.nan,'no_valid_trajectory',np.nan
    o=np.argsort(h); h=h[o]; y=y[o]
    h0=np.r_[0.0,h]; y0=np.r_[float(current_T),y]
    if crossed(phase,current_T,threshold) and direction_valid(phase,current_delta_prev):
        return 0.0,'label_matched_at_source',float(h[-1])
    ph,pt=h0[0],y0[0]
    for hh,tt in zip(h0[1:],y0[1:]):
        d=tt-pt
        if crossed(phase,tt,threshold) and direction_valid(phase,d):
            if not crossed(phase,pt,threshold): return linear_crossing(ph,pt,hh,tt,threshold),'label_matched_interpolated_crossing',float(h[-1])
            return float(hh),'label_matched_direction_at_horizon',float(h[-1])
        ph,pt=hh,tt
    # Transparent final-trend extrapolation, with mode-specific cap.
    n=min(EXTRAPOLATE_FINAL_N_POINTS,len(h0))
    if n>=3:
        slope,intercept=np.polyfit(h0[-n:],y0[-n:],1)
        good=(phase=='heating' and slope>EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN) or (phase=='cooling' and slope<-EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN)
        if good:
            hp=(float(threshold)-intercept)/slope
            cap=float(h0[-1]+EXTRAPOLATE_MAX_EXTRA_MIN_BY_PHASE[phase])
            if np.isfinite(hp) and hp>h0[-1]: return float(min(hp,cap)),('label_matched_extrapolated_final_trend' if hp<=cap else 'projected_beyond_cap_used_cap'),float(h[-1])
    return float(h0[-1]),'no_label_match_used_last_horizon',float(h[-1])

def source_to_anchor_remaining(source_crossing_min,asof_gap_min):
    if not np.isfinite(source_crossing_min): return np.nan
    return float(max(0.0,float(source_crossing_min)-float(asof_gap_min)))

# Deterministic self-test for the alignment correction.
assert abs(source_to_anchor_remaining(35.0,3.0)-32.0)<1e-12
print('Converter/alignment self-test passed.')


In [ ]:

# Direct room/mode Extra Trees remaining-duration baseline
#    Fit local models only with eligible training data.

DIRECT_INPUTS=[c for c in MODEL_INPUTS if c!='horizon_min']

# No score, duration label, room ID, or event boundary may enter predictors.
for bad in [
    'remaining_trajectory_quality_score','true_remaining_duration_min',
    'remaining_duration_min','room','event_id','event_key'
]:
    if bad in DIRECT_INPUTS:
        raise RuntimeError(f'Forbidden direct-duration input: {bad}')

DIRECT_PRED=np.full(len(ANCHORS),np.nan,dtype=float)
direct_manifest=[]
direct_support_rows=[]

for phase in PHASES:
    rooms=sorted(
        ANCHORS.loc[ANCHORS['phase']==phase,'room']
        .dropna().astype(str).unique()
    )

    for room in rooms:
        m=(ANCHORS['phase']==phase)&(ANCHORS['room']==room)
        tr=ANCHORS[m&(ANCHORS['split']=='train')].copy()
        va=ANCHORS[m&(ANCHORS['split']=='valid')].copy()
        te=ANCHORS[m&(ANCHORS['split']=='test')].copy()

        y_all=pd.to_numeric(
            tr['true_remaining_duration_min'],errors='coerce'
        ).to_numpy(float)
        good_y=np.isfinite(y_all)
        n_train_usable=int(good_y.sum())

        supported=n_train_usable>=MIN_DIRECT_ET_TRAIN_ANCHORS
        if supported:
            reason='supported'
        elif n_train_usable==0:
            reason='no_usable_train_anchors_under_authoritative_split'
        else:
            reason=f'usable_train_anchors_below_minimum_{MIN_DIRECT_ET_TRAIN_ANCHORS}'

        direct_support_rows.append({
            'phase':phase,
            'room':room,
            'supported':bool(supported),
            'reason':reason,
            'train_anchors_raw':int(len(tr)),
            'train_anchors_usable':n_train_usable,
            'valid_anchors':int(len(va)),
            'test_anchors':int(len(te)),
            'train_events':int(tr['event_key'].nunique()) if len(tr) else 0,
            'valid_events':int(va['event_key'].nunique()) if len(va) else 0,
            'test_events':int(te['event_key'].nunique()) if len(te) else 0,
        })

        if not supported:
            print(
                f'SKIP direct duration ET: {phase}/{room} | '
                f'usable TRAIN anchors={n_train_usable}, valid={len(va)}, '
                f'test={len(te)} | {reason}'
            )
            continue

        tr_fit=tr.loc[good_y].copy()
        y_fit=y_all[good_y]
        X_fit_num=tr_fit[DIRECT_INPUTS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
        # Prefer room median; if absent for this room, use Notebook-02's
        # training-only phase median for the same feature.
        for c in DIRECT_INPUTS:
            room_med=pd.to_numeric(X_fit_num[c],errors='coerce').median()
            fill_value=float(room_med) if np.isfinite(room_med) else float(PREP['phase_medians'][phase][c])
            X_fit_num[c]=X_fit_num[c].fillna(fill_value)
        imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X_fit_num)
        X_fit=imp.transform(X_fit_num)
        model=ExtraTreesRegressor(**DIRECT_ET_PARAMS).fit(X_fit,y_fit)

        idx=ANCHORS.index[m].to_numpy()
        X_all=imp.transform(
            ANCHORS.loc[idx,DIRECT_INPUTS].apply(pd.to_numeric,errors='coerce')
        )
        DIRECT_PRED[idx]=np.maximum(0.0,model.predict(X_all))

        p=MODEL_DIR/f'direct_ET_{phase}_{room}.joblib'
        joblib.dump({
            'model':model,
            'imputer':imp,
            'features':DIRECT_INPUTS,
            'phase':phase,
            'room':room,
            'train_anchors_usable':n_train_usable,
        },p)

        direct_manifest.append({
            'phase':phase,
            'room':room,
            'train_anchors_raw':int(len(tr)),
            'train_anchors_usable':n_train_usable,
            'artifact':str(p),
        })

ANCHORS['pred_remaining_Direct_RoomPhase_ET']=DIRECT_PRED
DIRECT_SUPPORTED_PAIRS={
    (str(r['phase']),str(r['room'])) for r in direct_manifest
}
ANCHORS['Direct_ET_supported']=[
    (str(p),str(r)) in DIRECT_SUPPORTED_PAIRS
    for p,r in zip(ANCHORS['phase'],ANCHORS['room'])
]

pd.DataFrame(
    direct_manifest,
    columns=['phase','room','train_anchors_raw','train_anchors_usable','artifact']
).to_csv(MODEL_DIR/'direct_ET_manifest.csv',index=False)

direct_support_df=pd.DataFrame(direct_support_rows)
direct_support_df.to_csv(MODEL_DIR/'direct_ET_support_audit.csv',index=False)
direct_support_df.to_csv(AUDIT_DIR/'direct_duration_ET_support_audit.csv',index=False)

print('\nDirect duration ET support audit:')
display(direct_support_df.sort_values(['phase','supported','room']).reset_index(drop=True))


In [ ]:
# Batch test-only trajectory inference while retaining the saved split and
# completion-time rules.

import time

def horizon_grid(phase):
    return np.arange(STEP_MIN, INDIRECT_MAX_HORIZON_BY_PHASE[phase]+1e-9, STEP_MIN, dtype=float)

def integrate_numpy_batch(h, pred_directed, T0, sign, v0_phys):
    pred_directed=np.asarray(pred_directed,float)
    T0=np.asarray(T0,float); sign=np.asarray(sign,float); v0_phys=np.asarray(v0_phys,float)
    pred_phys=sign[:,None]*pred_directed
    dt=np.r_[h[0],np.diff(h)].astype(float)
    prev=np.concatenate([v0_phys[:,None],pred_phys[:,:-1]],axis=1)
    return T0[:,None]+np.cumsum(0.5*dt[None,:]*(prev+pred_phys),axis=1)

def initial_rate_array(df):
    v=pd.to_numeric(df['Room_dTdt_now_bwd2_C_per_min'] if 'Room_dTdt_now_bwd2_C_per_min' in df.columns else pd.Series(np.nan,index=df.index),errors='coerce').to_numpy(float)
    fb=pd.to_numeric(df['Room_dTdt_central_-5min_C_per_min'] if 'Room_dTdt_central_-5min_C_per_min' in df.columns else pd.Series(0.0,index=df.index),errors='coerce').to_numpy(float)
    bad=~np.isfinite(v); v[bad]=fb[bad]; v[~np.isfinite(v)]=0.0
    return v

def make_expanded_anchor_frame(chunk,h):
    H=len(h)
    expanded=chunk.loc[chunk.index.repeat(H)].copy().reset_index(drop=True)
    expanded['horizon_min']=np.tile(h,len(chunk))
    expanded['target_timestamp']=pd.to_datetime(expanded['feature_source_timestamp'])+pd.to_timedelta(expanded['horizon_min'],unit='m')
    return expanded

EVAL_ANCHORS=ANCHORS[ANCHORS['split'].eq('test')].copy().reset_index(drop=True)
print('TEST-only duration trajectory inference:',len(EVAL_ANCHORS),'anchors')

records=[]
NN_ANCHOR_CHUNK_SIZE=1024
global_t0=time.perf_counter()

for phase in PHASES:
    phase_t0=time.perf_counter()
    ap=EVAL_ANCHORS[EVAL_ANCHORS['phase'].eq(phase)].copy().reset_index(drop=True)
    h=horizon_grid(phase); H=len(h)
    n_chunks=int(np.ceil(len(ap)/NN_ANCHOR_CHUNK_SIZE))
    print(f'\n{phase.upper()}: {len(ap)} TEST anchors | {H} horizons | {n_chunks} chunks')

    for chunk_no,lo in enumerate(range(0,len(ap),NN_ANCHOR_CHUNK_SIZE),start=1):
        t0=time.perf_counter()
        chunk=ap.iloc[lo:lo+NN_ANCHOR_CHUNK_SIZE].copy().reset_index(drop=True)
        B=len(chunk)
        expanded=make_expanded_anchor_frame(chunk,h)

        # Shared NN: one GPU call for B*H rows
        X=transform_nn(expanded)
        xt=torch.from_numpy(X).to(DEVICE,dtype=torch.float32,non_blocking=True)
        with torch.inference_mode():
            pred_directed=NN_MODEL(xt).float().reshape(B,H).cpu().numpy()
        del xt

        sign_arr=pd.to_numeric(chunk['thermal_mode_sign'],errors='coerce').to_numpy(float)
        T0_arr=pd.to_numeric(chunk['Room_T'],errors='coerce').to_numpy(float)
        v0_arr=initial_rate_array(chunk)
        pT_nn_all=integrate_numpy_batch(h,pred_directed,T0_arr,sign_arr,v0_arr)

        # Indirect ET: one predict call per room in this chunk
        pT_et_all=np.full((B,H),np.nan,dtype=float)
        et_supported=np.zeros(B,dtype=bool)
        rooms=chunk['room'].astype(str).to_numpy()

        for room in np.unique(rooms):
            art=ET_ART.get((phase,str(room)))
            if art is None:
                continue
            apos=np.flatnonzero(rooms==room)
            ridx=(apos[:,None]*H+np.arange(H)[None,:]).reshape(-1)
            syn=expanded.iloc[ridx]
            Xe=art['imputer'].transform(syn[art['feature_names']].apply(pd.to_numeric,errors='coerce'))
            delta=art['model'].predict(Xe).reshape(len(apos),H)
            pT_et_all[apos,:]=T0_arr[apos,None]+delta
            et_supported[apos]=True

        # Calculate completion time from predicted temperatures.
        for bi in range(B):
            row=chunk.iloc[bi]
            v0=float(v0_arr[bi]); current_delta=v0*STEP_MIN
            threshold=float(row['Duration_Crossing_Threshold_C'])

            cross_nn,ctype_nn,maxh_nn=estimate_crossing_from_source(phase,float(T0_arr[bi]),threshold,h,pT_nn_all[bi],current_delta)
            pred_nn=source_to_anchor_remaining(cross_nn,float(row['asof_gap_min']))
            common={'anchor_key':row['anchor_key'],'phase':phase,'room':row['room'],'event_key':row['event_key'],'split':'test','anchor_kind':row.get('anchor_kind','anchor'),'anchor_timestamp':row['anchor_timestamp'],'feature_source_timestamp':row['feature_source_timestamp'],'asof_gap_min':row['asof_gap_min'],'true_remaining_duration_min':row['true_remaining_duration_min'],'remaining_trajectory_quality_score':row['remaining_trajectory_quality_score']}
            records.append({**common,'method':'Unified_Derivative_NN','pred_remaining_duration_min':pred_nn,'crossing_from_feature_source_min':cross_nn,'crossing_type':ctype_nn,'max_horizon_min':maxh_nn})

            if et_supported[bi]:
                cross_et,ctype_et,maxh_et=estimate_crossing_from_source(phase,float(T0_arr[bi]),threshold,h,pT_et_all[bi],current_delta)
                pred_et=source_to_anchor_remaining(cross_et,float(row['asof_gap_min']))
            else:
                pred_et=np.nan; cross_et=np.nan; ctype_et='unsupported_no_room_phase_train_model'; maxh_et=float(h[-1])
            records.append({**common,'method':'Indirect_RoomPhase_ET','pred_remaining_duration_min':pred_et,'crossing_from_feature_source_min':cross_et,'crossing_type':ctype_et,'max_horizon_min':maxh_et})

        elapsed=time.perf_counter()-t0
        done=min(lo+B,len(ap))
        print(f'  {phase} chunk {chunk_no:02d}/{n_chunks:02d}: {done}/{len(ap)} anchors | {elapsed:.1f} s')

        # Keep partial work if Colab disconnects.
        pd.DataFrame(records).to_csv(PRED_DIR/'indirect_and_nn_duration_predictions_PARTIAL.csv',index=False)

    print(f'{phase.upper()} finished in {time.perf_counter()-phase_t0:.1f} s')

INDIRECT=pd.DataFrame(records)

# Coverage audit: shared NN must predict every TEST anchor exactly once.
expected=set(EVAL_ANCHORS['anchor_key'].astype(str))
actual=set(INDIRECT.loc[INDIRECT['method'].eq('Unified_Derivative_NN'),'anchor_key'].astype(str))
if expected!=actual:
    raise RuntimeError(f'Unified NN TEST coverage mismatch: missing={len(expected-actual)}, extra={len(actual-expected)}')
if not INDIRECT['split'].eq('test').all():
    raise RuntimeError('Optimized Notebook 03 inference must contain TEST predictions only.')

INDIRECT.to_csv(PRED_DIR/'indirect_and_nn_duration_predictions.csv',index=False)
partial=PRED_DIR/'indirect_and_nn_duration_predictions_PARTIAL.csv'
if partial.exists(): partial.unlink()

print('\nGenerated TEST-only GPU-batched NN + room-batched indirect-ET predictions:',len(INDIRECT))
print('Expected rows = 2 x TEST anchors =',2*len(EVAL_ANCHORS))
print('Total inference time:',f'{time.perf_counter()-global_t0:.1f} s')
display(INDIRECT.groupby(['phase','method'],as_index=False).agg(anchors=('anchor_key','nunique'),finite_predictions=('pred_remaining_duration_min',lambda s:int(pd.to_numeric(s,errors='coerce').notna().sum()))))


In [ ]:

# Evaluate duration errors and regularity diagnostics

direct_cols=['anchor_key','phase','room','event_key','split','anchor_kind','anchor_timestamp','feature_source_timestamp','asof_gap_min','true_remaining_duration_min','remaining_trajectory_quality_score','quality_bin','pred_remaining_Direct_RoomPhase_ET','Direct_ET_supported']
direct=ANCHORS[direct_cols].copy(); direct['method']='Direct_RoomPhase_ET'; direct['pred_remaining_duration_min']=direct['pred_remaining_Direct_RoomPhase_ET']; direct=direct.drop(columns='pred_remaining_Direct_RoomPhase_ET')
ALL=pd.concat([direct,INDIRECT],ignore_index=True,sort=False)
ALL['error_min']=pd.to_numeric(ALL['pred_remaining_duration_min'],errors='coerce')-pd.to_numeric(ALL['true_remaining_duration_min'],errors='coerce')
ALL['absolute_error_min']=ALL['error_min'].abs()
ALL['relative_error_denom_min']=np.maximum(pd.to_numeric(ALL['true_remaining_duration_min'],errors='coerce'),STEP_MIN)
ALL['relative_absolute_error_pct']=100.0*ALL['absolute_error_min']/ALL['relative_error_denom_min']
ALL['relative_signed_error_pct']=100.0*ALL['error_min']/ALL['relative_error_denom_min']
ALL['quality_bin']=pd.cut(pd.to_numeric(ALL['remaining_trajectory_quality_score'],errors='coerce'),[-1e-9,.25,.5,.75,1.000001],labels=['q<=0.25','0.25<q<=0.50','0.50<q<=0.75','q>0.75'])
ALL.to_csv(PRED_DIR/'ALL_remaining_duration_predictions_with_relative_errors.csv',index=False)

def duration_metrics(g):
    d=g[['error_min','absolute_error_min','relative_absolute_error_pct','relative_signed_error_pct']].apply(pd.to_numeric,errors='coerce').dropna()
    if len(d)==0: return pd.Series({'n':0,'MAE_min':np.nan,'RMSE_min':np.nan,'Bias_min':np.nan,'MedianAE_min':np.nan,'P90AE_min':np.nan,'MeanRAE_pct':np.nan,'MedianRAE_pct':np.nan,'P90RAE_pct':np.nan,'RelativeBias_pct':np.nan})
    e=d['error_min'].to_numpy(float); ae=d['absolute_error_min'].to_numpy(float); rae=d['relative_absolute_error_pct'].to_numpy(float); rse=d['relative_signed_error_pct'].to_numpy(float)
    return pd.Series({'n':len(d),'MAE_min':np.mean(ae),'RMSE_min':np.sqrt(np.mean(e**2)),'Bias_min':np.mean(e),'MedianAE_min':np.median(ae),'P90AE_min':np.percentile(ae,90),
                      'MeanRAE_pct':np.mean(rae),'MedianRAE_pct':np.median(rae),'P90RAE_pct':np.percentile(rae,90),'RelativeBias_pct':np.mean(rse)})

TEST=ALL[ALL['split'].eq('test')].copy(); METHODS=['Direct_RoomPhase_ET','Indirect_RoomPhase_ET','Unified_Derivative_NN']
# Exact common anchors: identical TEST split and a finite prediction from every compared
# method.
wide=TEST.pivot_table(index='anchor_key',columns='method',values='pred_remaining_duration_min',aggfunc='first')
for m in METHODS:
    if m not in wide.columns: wide[m]=np.nan
common_keys=set(wide.dropna(subset=METHODS).index.astype(str)); TEST_COMMON=TEST[TEST['anchor_key'].astype(str).isin(common_keys)].copy()
MAIN=TEST_COMMON.groupby(['phase','method'],as_index=False).apply(duration_metrics,include_groups=False).reset_index(drop=True)
MAIN.to_csv(TABLE_DIR/'PAPER_MAIN_duration_metrics_EXACT_COMMON_SUPPORT.csv',index=False)
NN_FULL=TEST[TEST['method'].eq('Unified_Derivative_NN')].copy(); NN_FULL_METRICS=NN_FULL.groupby(['phase','method'],as_index=False).apply(duration_metrics,include_groups=False).reset_index(drop=True)
NN_FULL_METRICS.to_csv(TABLE_DIR/'PAPER_UnifiedNN_duration_metrics_FULL_TEST_SUPPORT.csv',index=False)

# q-stratified and true-duration-stratified results.
QB=TEST_COMMON.groupby(['phase','quality_bin','method'],observed=True,as_index=False).apply(duration_metrics,include_groups=False).reset_index(drop=True); QB.to_csv(TABLE_DIR/'PAPER_duration_metrics_by_q_bin_COMMON_SUPPORT.csv',index=False)
TEST_COMMON['true_remaining_bin']=pd.cut(pd.to_numeric(TEST_COMMON['true_remaining_duration_min'],errors='coerce'),[0,10,20,40,60,np.inf],include_lowest=True,labels=['0–10','10–20','20–40','40–60','>60'])
TB=TEST_COMMON.groupby(['phase','true_remaining_bin','method'],observed=True,as_index=False).apply(duration_metrics,include_groups=False).reset_index(drop=True); TB.to_csv(TABLE_DIR/'PAPER_duration_metrics_by_true_remaining_bin_COMMON_SUPPORT.csv',index=False)
ROOM=TEST_COMMON.groupby(['phase','room','method'],as_index=False).apply(duration_metrics,include_groups=False).reset_index(drop=True); ROOM.to_csv(TABLE_DIR/'PAPER_duration_metrics_by_room_COMMON_SUPPORT.csv',index=False)
KIND=TEST_COMMON.groupby(['phase','anchor_kind','method'],as_index=False).apply(duration_metrics,include_groups=False).reset_index(drop=True); KIND.to_csv(TABLE_DIR/'PAPER_duration_metrics_by_anchor_kind_COMMON_SUPPORT.csv',index=False)

print('PRIMARY REMAINING-TIME RESULTS — EXACT COMMON TEST SUPPORT')
display(MAIN.round(3)); print('\nUNIFIED NN — FULL TEST SUPPORT'); display(NN_FULL_METRICS.round(3)); print('\nERROR BY q BIN'); display(QB.round(3))

# q distribution statistics and figures from saved anchors.
QSTATS=(ANCHORS.groupby('phase')['remaining_trajectory_quality_score'].agg(['count','mean','std','median',lambda x:x.quantile(.25),lambda x:x.quantile(.75)]).reset_index())
QSTATS.columns=['phase','n','mean_q','std_q','median_q','q25','q75']; QSTATS.to_csv(TABLE_DIR/'PAPER_q_distribution_summary_by_phase.csv',index=False)
qbin_counts=(ANCHORS.groupby(['phase','quality_bin'],observed=True).size().rename('anchors').reset_index()); qbin_counts['fraction']=qbin_counts.groupby('phase')['anchors'].transform(lambda x:x/x.sum()); qbin_counts.to_csv(TABLE_DIR/'PAPER_q_bin_distribution_by_phase.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,4.8),constrained_layout=True)
for ax,phase in zip(axes,PHASES):
    q=pd.to_numeric(ANCHORS.loc[ANCHORS['phase'].eq(phase),'remaining_trajectory_quality_score'],errors='coerce').dropna(); ax.hist(q,bins=np.linspace(0,1,21),density=True,alpha=.75)
    ax.axvline(q.median(),linestyle='--',label=f'Median q={q.median():.2f}'); ax.set_xlim(0,1); ax.set(title=phase.capitalize(),xlabel='Trajectory-regularity score q',ylabel='Density'); ax.grid(alpha=.2); ax.legend(frameon=False)
fig.savefig(FIG_DIR/'PAPER_q_distribution_heating_cooling.png',dpi=300,bbox_inches='tight'); plt.show()

# Spearman q vs error for final NN.
spear=[]
for phase,g in NN_FULL.groupby('phase'):
    z=g[['remaining_trajectory_quality_score','absolute_error_min','relative_absolute_error_pct']].apply(pd.to_numeric,errors='coerce').dropna()
    rho_ae,p_ae=spearmanr(z['remaining_trajectory_quality_score'],z['absolute_error_min']); rho_rel,p_rel=spearmanr(z['remaining_trajectory_quality_score'],z['relative_absolute_error_pct'])
    spear.append({'phase':phase,'n':len(z),'rho_q_vs_absolute_error':rho_ae,'p_absolute':p_ae,'rho_q_vs_relative_error':rho_rel,'p_relative':p_rel})
SPEAR=pd.DataFrame(spear); SPEAR.to_csv(TABLE_DIR/'PAPER_q_vs_duration_error_spearman.csv',index=False); print('\nq vs NN error Spearman:'); display(SPEAR.round(4))

# FIGURE 1 — headline absolute + relative error comparison.
fig,axes=plt.subplots(2,2,figsize=(14,9),constrained_layout=True)
for r,phase in enumerate(PHASES):
    d=MAIN[MAIN['phase'].eq(phase)].copy(); x=np.arange(len(d)); labels=[m.replace('_','\n') for m in d['method']]
    axes[r,0].bar(x,d['MAE_min']); axes[r,0].set_xticks(x,labels); axes[r,0].set(title=f'{phase.capitalize()} — absolute error',ylabel='MAE (min)'); axes[r,0].grid(axis='y',alpha=.25)
    axes[r,1].bar(x,d['MedianRAE_pct']); axes[r,1].set_xticks(x,labels); axes[r,1].set(title=f'{phase.capitalize()} — duration-relative error',ylabel='Median RAE (%)'); axes[r,1].grid(axis='y',alpha=.25)
fig.savefig(FIG_DIR/'PAPER_remaining_time_method_comparison_absolute_and_relative.png',dpi=300,bbox_inches='tight'); plt.show()

# FIGURE 2 — parity plots colored by q; q is visible and interpretable.
fig,axes=plt.subplots(2,3,figsize=(16,9),constrained_layout=True,sharex='row',sharey='row')
_sc=None
for r,phase in enumerate(PHASES):
    for c,method in enumerate(METHODS):
        ax=axes[r,c]; g=TEST_COMMON[(TEST_COMMON['phase']==phase)&(TEST_COMMON['method']==method)].dropna(subset=['pred_remaining_duration_min','true_remaining_duration_min','remaining_trajectory_quality_score'])
        _sc=ax.scatter(g['true_remaining_duration_min'],g['pred_remaining_duration_min'],c=g['remaining_trajectory_quality_score'],cmap='viridis',vmin=0,vmax=1,s=18,alpha=.75)
        lim=max(float(g['true_remaining_duration_min'].max()),float(g['pred_remaining_duration_min'].max())) if len(g) else 1.; ax.plot([0,lim],[0,lim],linestyle='--'); ax.set(title=f'{phase.capitalize()}\n{method.replace("_"," ")}',xlabel='True remaining time (min)',ylabel='Predicted remaining time (min)'); ax.grid(alpha=.2)
fig.colorbar(_sc,ax=axes.ravel().tolist(),label='Trajectory regularity q',shrink=.8)
fig.savefig(FIG_DIR/'PAPER_remaining_time_parity_colored_by_q.png',dpi=300,bbox_inches='tight'); plt.show()

# FIGURE 3 — relative error versus true duration colored by q.
fig,axes=plt.subplots(2,3,figsize=(16,9),constrained_layout=True,sharex='row')
_sc=None
for r,phase in enumerate(PHASES):
    for c,method in enumerate(METHODS):
        ax=axes[r,c]; g=TEST_COMMON[(TEST_COMMON['phase']==phase)&(TEST_COMMON['method']==method)].dropna(subset=['true_remaining_duration_min','relative_absolute_error_pct','remaining_trajectory_quality_score'])
        _sc=ax.scatter(g['true_remaining_duration_min'],g['relative_absolute_error_pct'],c=g['remaining_trajectory_quality_score'],cmap='viridis',vmin=0,vmax=1,s=18,alpha=.75)
        ax.set(title=f'{phase.capitalize()}\n{method.replace("_"," ")}',xlabel='True remaining time (min)',ylabel='Relative absolute error (%)'); ax.grid(alpha=.2)
fig.colorbar(_sc,ax=axes.ravel().tolist(),label='Trajectory regularity q',shrink=.8)
fig.savefig(FIG_DIR/'PAPER_relative_duration_error_vs_true_colored_by_q.png',dpi=300,bbox_inches='tight'); plt.show()

# FIGURE 4 — errors by q bin, absolute and relative.
q_order=['q<=0.25','0.25<q<=0.50','0.50<q<=0.75','q>0.75']
fig,axes=plt.subplots(2,2,figsize=(15,9),constrained_layout=True)
for r,phase in enumerate(PHASES):
    d=QB[QB['phase'].eq(phase)].copy()
    for method,g in d.groupby('method'):
        g=g.set_index('quality_bin').reindex(q_order).reset_index(); axes[r,0].plot(q_order,g['MAE_min'],marker='o',label=method); axes[r,1].plot(q_order,g['MedianRAE_pct'],marker='o',label=method)
    axes[r,0].set(title=f'{phase.capitalize()} — absolute error by q',ylabel='MAE (min)'); axes[r,1].set(title=f'{phase.capitalize()} — relative error by q',ylabel='Median RAE (%)')
    for ax in axes[r]: ax.tick_params(axis='x',rotation=20); ax.grid(alpha=.25); ax.legend(frameon=False)
fig.savefig(FIG_DIR/'PAPER_duration_error_by_q_bin_methods.png',dpi=300,bbox_inches='tight'); plt.show()

# FIGURE 5 — performance by true remaining-time range.
true_order=['0–10','10–20','20–40','40–60','>60']
fig,axes=plt.subplots(2,2,figsize=(15,9),constrained_layout=True)
for r,phase in enumerate(PHASES):
    d=TB[TB['phase'].eq(phase)].copy()
    for method,g in d.groupby('method'):
        g=g.set_index('true_remaining_bin').reindex(true_order).reset_index(); axes[r,0].plot(true_order,g['MAE_min'],marker='o',label=method); axes[r,1].plot(true_order,g['MedianRAE_pct'],marker='o',label=method)
    axes[r,0].set(title=f'{phase.capitalize()} — absolute error vs remaining-time range',ylabel='MAE (min)'); axes[r,1].set(title=f'{phase.capitalize()} — relative error vs remaining-time range',ylabel='Median RAE (%)')
    for ax in axes[r]: ax.tick_params(axis='x',rotation=20); ax.grid(alpha=.25); ax.legend(frameon=False)
fig.savefig(FIG_DIR/'PAPER_duration_error_by_true_remaining_range.png',dpi=300,bbox_inches='tight'); plt.show()

# Crossing-type diagnostics for the final NN and indirect ET.
CROSS=TEST[TEST['method'].isin(['Unified_Derivative_NN','Indirect_RoomPhase_ET'])].groupby(['phase','method','crossing_type'],dropna=False,as_index=False).apply(duration_metrics,include_groups=False).reset_index(drop=True)
counts=TEST[TEST['method'].isin(['Unified_Derivative_NN','Indirect_RoomPhase_ET'])].groupby(['phase','method','crossing_type'],dropna=False).size().rename('count').reset_index(); CROSS=CROSS.merge(counts,on=['phase','method','crossing_type'],how='left'); CROSS.to_csv(TABLE_DIR/'PAPER_crossing_type_counts_and_errors.csv',index=False)


In [ ]:

# End-region temperature/rate diagnostics + publication T-t examples

# Notebook 02 saved selected-NN predictions on the long-format trajectory universe.
temp_pred_path=TRAIN_RUN/'predictions'/'temperature_predictions_wide.csv'
TP=pd.read_csv(temp_pred_path)
for c in ['timestamp','target_timestamp','rampup_end_time']:
    if c in TP.columns: TP[c]=pd.to_datetime(TP[c],errors='coerce')
for c in ['target_Room_T','pred_T_UnifiedNN','pred_T_RoomPhase_ExtraTrees','target_directed_rate_C_per_min','pred_directed_rate_UnifiedNN','Phase_Target_T_C','thermal_mode_sign']:
    if c in TP.columns: TP[c]=pd.to_numeric(TP[c],errors='coerce')
TP['Duration_Crossing_Threshold_C']=TP['Phase_Target_T_C']-TP['thermal_mode_sign']*END_BAND_C
TP['minutes_to_true_end']=(TP['rampup_end_time']-TP['target_timestamp']).dt.total_seconds()/60.0
TP['temperature_error_NN_C']=TP['pred_T_UnifiedNN']-TP['target_Room_T']
TP['directed_rate_error_C_per_min']=TP['pred_directed_rate_UnifiedNN']-TP['target_directed_rate_C_per_min']
TP['distance_truth_to_end_threshold_C']=(TP['target_Room_T']-TP['Duration_Crossing_Threshold_C']).abs()

# Mutually exclusive event-end regions.
def end_region(r):
    ctx=str(r.get('target_context','')).lower(); m=r['minutes_to_true_end']
    if ctx.startswith('post_end') or (np.isfinite(m) and m<0): return 'post_end'
    if np.isfinite(m) and 0<=m<=15: return 'pre_end_0_15'
    if np.isfinite(m) and 15<m<=30: return 'pre_end_15_30'
    if np.isfinite(m) and m>30: return 'pre_end_gt30'
    return 'other'
TP['end_region']=TP.apply(end_region,axis=1)

rows=[]
for (phase,region),g in TP[TP['split'].eq('test')].groupby(['phase','end_region']):
    et=pd.to_numeric(g['temperature_error_NN_C'],errors='coerce').dropna().to_numpy(float)
    er=pd.to_numeric(g['directed_rate_error_C_per_min'],errors='coerce').dropna().to_numpy(float)
    rows.append({'phase':phase,'region':region,'n_T':len(et),'T_MAE_C':float(np.mean(np.abs(et))) if len(et) else np.nan,'T_RMSE_C':float(np.sqrt(np.mean(et**2))) if len(et) else np.nan,'T_Bias_C':float(np.mean(et)) if len(et) else np.nan,'n_rate':len(er),'rate_MAE_C_per_min':float(np.mean(np.abs(er))) if len(er) else np.nan,'rate_RMSE_C_per_min':float(np.sqrt(np.mean(er**2))) if len(er) else np.nan})
END_REGION=pd.DataFrame(rows); END_REGION.to_csv(TABLE_DIR/'FINAL_UnifiedNN_temperature_rate_metrics_by_end_region.csv',index=False)

near=TP[TP['split'].eq('test') & (TP['distance_truth_to_end_threshold_C']<=0.5)].copy()
near_rows=[]
for phase,g in near.groupby('phase'):
    et=pd.to_numeric(g['temperature_error_NN_C'],errors='coerce').dropna().to_numpy(float); er=pd.to_numeric(g['directed_rate_error_C_per_min'],errors='coerce').dropna().to_numpy(float)
    near_rows.append({'phase':phase,'n':len(g),'T_MAE_C':float(np.mean(np.abs(et))) if len(et) else np.nan,'T_RMSE_C':float(np.sqrt(np.mean(et**2))) if len(et) else np.nan,'rate_MAE_C_per_min':float(np.mean(np.abs(er))) if len(er) else np.nan})
NEAR_THRESHOLD=pd.DataFrame(near_rows); NEAR_THRESHOLD.to_csv(TABLE_DIR/'FINAL_UnifiedNN_metrics_within_0p5C_of_end_threshold.csv',index=False)
print('END-REGION NN TEMPERATURE/RATE DIAGNOSTICS:'); display(END_REGION.round(4))
print('\nWITHIN 0.5 C OF TRUE END THRESHOLD:'); display(NEAR_THRESHOLD.round(4))

# Re-load long-format truth for T-t plots.
long_frames=[]
for phase in PHASES:
    p=DATA_DIR/phase/f'{phase}_full_rampup_5min_samples_LONG.csv'; d=pd.read_csv(p); d['phase']=phase
    for c in ['timestamp','target_timestamp']: d[c]=pd.to_datetime(d[c],errors='coerce')
    d['event_id']=d['event_id'].map(norm_event_id); d['room']=d['room'].astype(str); d['event_key']=phase+'|'+d['room']+'|'+d['event_id']; long_frames.append(d)
LONG=pd.concat(long_frames,ignore_index=True,sort=False)

# Reconstruct plot trajectories with the same rate-integration procedure used in
# evaluation.
def integrate_numpy_plot(h, pred_directed, T0, sign, v0_phys):
    out = integrate_numpy_batch(
        np.asarray(h, dtype=float),
        np.asarray(pred_directed, dtype=float)[None, :],
        np.asarray([T0], dtype=float),
        np.asarray([sign], dtype=float),
        np.asarray([v0_phys], dtype=float),
    )
    return out[0]

# This keeps the plotting cell independent of earlier evaluation helpers.
def repeat_anchor_for_horizons_plot(row, phase):
    h = horizon_grid(phase)
    base = pd.DataFrame([row.to_dict()] * len(h))
    base['horizon_min'] = h
    base['target_timestamp'] = (
        pd.Timestamp(row['feature_source_timestamp'])
        + pd.to_timedelta(h, unit='m')
    )
    return base

# Best/median/worst inside-grid anchors for the selected NN.
for phase in PHASES:
    pool=NN_FULL[(NN_FULL['phase']==phase)&(NN_FULL['anchor_kind'].astype(str).eq('inside_grid'))].copy().sort_values('absolute_error_min')
    if len(pool)<3: pool=NN_FULL[NN_FULL['phase']==phase].copy().sort_values('absolute_error_min')
    if pool.empty: continue
    picks={'best':pool.iloc[0],'median':pool.iloc[len(pool)//2],'worst':pool.iloc[-1]}
    for label,predrow in picks.items():
        a=ANCHORS[ANCHORS['anchor_key'].astype(str).eq(str(predrow['anchor_key']))]
        if a.empty: continue
        row=a.iloc[0]; syn=repeat_anchor_for_horizons_plot(row,phase); h=horizon_grid(phase); Xn=transform_nn(syn)
        with torch.no_grad(): pv=NN_MODEL(torch.as_tensor(Xn,dtype=torch.float32,device=DEVICE)).cpu().numpy()
        v0=pd.to_numeric(row.get('Room_dTdt_now_bwd2_C_per_min',0),errors='coerce'); v0=float(v0) if np.isfinite(v0) else 0.0
        pT_nn=integrate_numpy_plot(h,pv,float(row['Room_T']),float(row['thermal_mode_sign']),v0)
        g=LONG[LONG['event_key']==row['event_key']].copy(); t0=pd.Timestamp(row['feature_source_timestamp']); truth=g[g['timestamp']==t0].sort_values('horizon_min')
        if truth.empty: continue
        ht=pd.to_numeric(truth['horizon_min'],errors='coerce').to_numpy(float); Tt=pd.to_numeric(truth['target_Room_T'],errors='coerce').to_numpy(float); vv=np.isfinite(ht)&np.isfinite(Tt); ht=ht[vv]; Tt=Tt[vv]
        if len(ht)==0: continue
        hmax=float(np.max(ht)); pm=h<=hmax+1e-9
        art=ET_ART.get((phase,str(row['room']))); pT_et=None
        if art is not None:
            Xe=art['imputer'].transform(syn[art['feature_names']].apply(pd.to_numeric,errors='coerce')); pT_et=float(row['Room_T'])+art['model'].predict(Xe)
        fig,ax=plt.subplots(figsize=(10.4,5.8))
        ax.plot([0.0]+ht.tolist(),[float(row['Room_T'])]+Tt.tolist(),marker='o',label='Observed T')
        ax.plot([0.0]+h[pm].tolist(),[float(row['Room_T'])]+pT_nn[pm].tolist(),linewidth=2,label='Unified derivative NN')
        if pT_et is not None: ax.plot([0.0]+h[pm].tolist(),[float(row['Room_T'])]+pT_et[pm].tolist(),linewidth=2,label='Indirect room/phase ET')
        ax.axhline(float(row['Duration_Crossing_Threshold_C']),linestyle='--',label='Event-end threshold')
        true_x=float(row['true_remaining_duration_min'])+float(row['asof_gap_min']); ax.axvline(true_x,linestyle='-',alpha=.7,label='Actual end')
        this=TEST[TEST['anchor_key'].astype(str).eq(str(row['anchor_key']))]
        for method,lbl in [('Direct_RoomPhase_ET','Direct ET end'),('Indirect_RoomPhase_ET','Indirect ET end'),('Unified_Derivative_NN','Unified NN end')]:
            z=this[this['method']==method]
            if z.empty: continue
            rem=pd.to_numeric(z.iloc[0]['pred_remaining_duration_min'],errors='coerce')
            if np.isfinite(rem): ax.axvline(float(rem)+float(row['asof_gap_min']),linestyle=':',alpha=.8,label=lbl)
        ax.set_xlim(0.0,min(hmax,PAPER_TEMPERATURE_XMAX_BY_PHASE[phase])); ax.set(title=f'{phase.capitalize()} — {label} selected-NN duration case | q={float(row["remaining_trajectory_quality_score"]):.2f}',xlabel='Minutes from feature-source time',ylabel='Room temperature (°C)'); ax.grid(alpha=.25); ax.legend(frameon=False,fontsize=8)
        fig.savefig(FIG_DIR/f'{phase}_FINAL_Tt_{label}.png',dpi=300,bbox_inches='tight'); plt.close(fig)
print('Saved FINAL best/median/worst T-t examples.')


In [ ]:

# Reference evaluation manifest

manifest={
    'notebook':'03_predict_remaining_duration_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL',
    'data_dir':str(DATA_DIR),'training_run':str(TRAIN_RUN),'output_dir':str(OUT_ROOT),'shared_split_sha256':CURRENT_SPLIT_SHA256,
    'same_chronological_split_for_all_models':True,'test_used_for_model_selection':False,'selected_score_mode':'score_weighted',
    'methods':['Direct_RoomPhase_ET','Indirect_RoomPhase_ET','Unified_Derivative_NN'],
    'primary_endpoint':'remaining duration','absolute_metrics':['MAE_min','RMSE_min','Bias_min','MedianAE_min','P90AE_min'],
    'relative_metric_definition':'RAE_pct = 100*abs(pred-true)/max(true,5 min)','relative_metrics':['MeanRAE_pct','MedianRAE_pct','P90RAE_pct','RelativeBias_pct'],
    'quality_score_reporting':'q distribution by phase; q-bin errors; parity/error plots colored by q; q shown on T-t examples; Spearman q-vs-error',
    'fair_comparison':'exact common TEST anchors after applying the one shared chronological split; unsupported room-specific ET predictions remain NaN rather than changing the split',
    'paper_tables':{
        'main_duration':str(TABLE_DIR/'PAPER_MAIN_duration_metrics_EXACT_COMMON_SUPPORT.csv'),
        'nn_full_duration':str(TABLE_DIR/'PAPER_UnifiedNN_duration_metrics_FULL_TEST_SUPPORT.csv'),
        'q_bins':str(TABLE_DIR/'PAPER_duration_metrics_by_q_bin_COMMON_SUPPORT.csv'),
        'true_remaining_bins':str(TABLE_DIR/'PAPER_duration_metrics_by_true_remaining_bin_COMMON_SUPPORT.csv'),
        'q_distribution':str(TABLE_DIR/'PAPER_q_distribution_summary_by_phase.csv'),
        'q_spearman':str(TABLE_DIR/'PAPER_q_vs_duration_error_spearman.csv'),
        'crossing_types':str(TABLE_DIR/'PAPER_crossing_type_counts_and_errors.csv'),
    }
}
path=OUT_ROOT/'duration_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'
with open(path,'w') as f: json.dump(manifest,f,indent=2,default=str)
print('Notebook 03 PAPER FINAL complete.')
print('Manifest:',path)
